In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED      = 42
N_TRIALS  = 20
COMPARER  = True
LGBM_BASE = dict(random_state=SEED, n_jobs=-1, verbose=-1,
                 importance_type='gain')

TARGETS = {
    'total'      : 'out.electricity.total.energy_consumption..kwh',
    'chauffage'  : 'out.electricity.heating.energy_consumption..kwh',
    'clim'       : 'out.electricity.cooling.energy_consumption..kwh',
    'eau_chaude' : 'out.electricity.hot_water.energy_consumption..kwh',
}
TWEEDIE = {'chauffage', 'clim'}
REEL, PREDIT, NEUTRE = '#2a78d6', '#eb6834', '#8C9CA3'


def to_numpy_dtypes(df):
    for col in df.columns:
        dt = df[col].dtype
        if hasattr(dt, 'numpy_dtype'):
            df[col] = df[col].astype(dt.numpy_dtype)
        elif hasattr(dt, 'pyarrow_dtype'):
            df[col] = df[col].astype(str(dt.pyarrow_dtype))
    return df


def params(usage, best=None, arbres=10_000):
    p = dict(n_estimators=arbres, **LGBM_BASE)
    p.update(best or {})
    if usage in TWEEDIE:
        p.update(objective='tweedie', tweedie_variance_power=1.4)
    return p


def entrainer(usage, cols, best=None, arbres=10_000):
    m = lgb.LGBMRegressor(**params(usage, best, arbres))
    m.fit(X_train[cols], Yt_train[usage], eval_set=[(X_val[cols], Yt_val[usage])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m, np.clip(m.predict(X_test[cols]), 0, None)

In [ ]:
import re

FT2   = 0.0929
RUS   = 0.1761
UUS   = 5.678
RFILM = 0.85
ISO   = dict(leger=110, moyen=165, lourd=260, tres_lourd=370)

def _U(Rus):
    return 1.0 / ((np.maximum(Rus, 0) + RFILM) * RUS)

def _poids(az):
    az %= 360
    if 135 <= az <= 225:
        return 1.0
    if az < 45 or az >= 315:
        return 0.3
    return 0.6

def build_aggregates(Xs, Rs):
    A_wall  = Rs['out.params.wall_area_above_grade_exterior..ft2'].values * FT2
    A_roof  = Rs['out.params.roof_area..ft2'].values                     * FT2
    A_floor = Rs['out.params.floor_area_lighting..ft2'].values           * FT2
    A_win   = Rs['out.params.window_area..ft2'].values                   * FT2
    A_door  = Rs['out.params.door_area..ft2'].values                     * FT2

    is_attic = Rs['in.geometry_attic_type'].astype(str).isin(['Vented Attic', 'Unvented Attic']).values
    R_top    = np.where(is_attic, Xs['in.insulation_ceiling'].values, Xs['in.insulation_roof'].values)
    UA = (_U(Xs['in.insulation_wall'].values) * A_wall
          + _U(R_top)                          * A_roof
          + _U(Xs['in.insulation_floor'].values) * A_floor
          + Xs['in.window_ufactor'].values * UUS * A_win
          + 1.14                                 * A_door)

    V    = A_floor * 2.5
    H_ve = 0.34 * (Xs['in.air_leakage_to_outside_ach50'].values / 20.0) * V

    wt  = Rs['in.geometry_wall_type'].astype(str)
    fin = Rs['in.geometry_wall_exterior_finish'].astype(str)
    cls = np.select(
        [wt.str.contains('Concrete').values, fin.str.contains('Brick').values,
         (wt == 'Wood Frame').values, (wt == 'Steel Frame').values],
        [ISO['tres_lourd'], ISO['lourd'], ISO['leger'], ISO['moyen']], default=ISO['moyen'])
    C = cls * A_floor

    shgc = Xs['in.window_shgc'].values
    ang  = np.degrees(np.arctan2(Xs['in.orientation_sin'].values,
                                 Xs['in.orientation_cos'].values)) % 360
    wa   = Rs['in.window_areas'].astype(str).values
    Asol = np.zeros(len(Xs))
    for i, (s, a, sh, aw) in enumerate(zip(wa, ang, shgc, A_win)):
        d = {'F': 0., 'B': 0., 'L': 0., 'R': 0.}
        for m in re.finditer(r'([FBLR])(\d+)', s):
            d[m.group(1)] = float(m.group(2))
        tot  = sum(d.values()) or 1.0
        dirs = {'F': a, 'R': a + 90, 'B': a + 180, 'L': a + 270}
        Asol[i] = sh * 0.9 * aw * sum((d[k] / tot) * _poids(dirs[k]) for k in d)

    comp = (A_wall + A_roof + A_floor + A_win + A_door) / np.maximum(V, 1)

    return pd.DataFrame({'UA': UA, 'H_ve': H_ve, 'C': C, 'A_solaire': Asol, 'compacite': comp},
                        index=Xs.index)

In [ ]:
def build_hvac_dse(Xs):
    loc  = Xs['in.duct_location_int'].values
    leak = Xs['in.duct_leakage'].values
    rins = Xs['in.duct_insulation'].values

    conditionne      = np.isin(loc, [0, 1, 4])
    perte_fuite      = leak
    perte_conduction = 0.10 / (1.0 + rins / 4.0)
    dse = np.where(conditionne, 1.0,
                   np.clip(1.0 - (perte_fuite + perte_conduction), 0.5, 1.0))
    return pd.Series(dse, index=Xs.index, name='DSE')

In [ ]:
ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW       = ROOT / 'data' / 'raw'

X    = pd.read_parquet(DATA_PROCESSED / 'X.parquet')
meta = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')

RAW_COLS = ['out.params.wall_area_above_grade_exterior..ft2', 'out.params.roof_area..ft2',
            'out.params.floor_area_lighting..ft2', 'out.params.window_area..ft2',
            'out.params.door_area..ft2', 'in.geometry_attic_type', 'in.geometry_wall_type',
            'in.geometry_wall_exterior_finish', 'in.window_areas']
raw = pd.read_parquet(DATA_RAW / 'upgrade0.parquet', columns=RAW_COLS)

Yt = pd.read_parquet(DATA_RAW / 'upgrade0.parquet', columns=list(TARGETS.values()))
Yt = Yt.rename(columns={v: k for k, v in TARGETS.items()})

DROP = [c for c in X.columns if any(k in c for k in [
    'insulation_', 'slab_', 'window_ufactor', 'window_shgc', 'window_front', 'air_leakage',
    'wall_exterior_finish_r', 'wall_finish_', 'roof_material', 'geometry_floor_area',
    'geometry_stories', 'geometry_foundation_type', 'geometry_garage', 'orientation_',
    'horiz_loc_', 'neighbor_'])]
DROP_SOCIO = ['in.area_median_income', 'in.state_metro_median_income', 'in.tenure',
              'in.household_has_tribal_persons', 'in.aiannh_area']
DROP_DHW = ['in.water_heater_in_unit', 'in.water_heater_technology_indirect',
            'in.water_heater_technology_storage', 'in.water_heater_technology_tankless',
            'in.water_heater_fuel_fuel_oil', 'in.water_heater_fuel_natural_gas',
            'in.water_heater_fuel_propane', 'in.water_heater_location_unconditioned']
DROP_USAGES = ['in.hot_tub_electric', 'in.has_hot_tub', 'in.misc_hot_tub_gas',
               'in.has_pool', 'in.pool_heater_present', 'in.pool_heater_electric',
               'in.pool_heater_gas', 'in.misc_gas_fireplace_present',
               'in.misc_gas_grill_present', 'in.misc_gas_lighting_present',
               'in.has_well_pump', 'in.has_ceiling_fan', 'in.ceiling_fan_used',
               'in.clothes_dryer_gas', 'in.clothes_dryer_electric', 'in.clothes_dryer_has',
               'in.clothes_washer_has', 'in.refrigerator_has', 'in.refrigerator_usage_level',
               'in.misc_extra_refrigerator_has']
DROP_HVAC_DUCTS = ['in.hvac_has_ducts', 'in.duct_location_int', 'in.duct_leakage',
                   'in.duct_insulation']
DROP = DROP + [c for c in DROP_SOCIO + DROP_DHW + DROP_USAGES + DROP_HVAC_DUCTS
               if c in X.columns]
KEEP = [c for c in X.columns if c not in DROP]

mask = (
    (meta['in.geometry_building_type_recs'] == 'Single-Family Detached') &
    (X['in.geometry_stories'] == 1)                                      &
    (meta['in.heating_fuel']  == 'Electricity')                          &
    (X['in.electric_vehicle_charger'] == 0)                              &
    (X['in.has_pool'] == 0)                                              &
    (X['in.has_pv']   == 0)
).values

agg   = build_aggregates(X[mask], raw[mask]).reset_index(drop=True)
dse   = build_hvac_dse(X[mask]).reset_index(drop=True)
keep  = X[mask][KEEP].reset_index(drop=True)
X_sub = to_numpy_dtypes(pd.concat([agg, dse, keep], axis=1))
Yt_sub = to_numpy_dtypes(Yt[mask].reset_index(drop=True).copy())
ashrae_sub = meta[mask]['in.ashrae_iecc_climate_zone_2004'].astype(str).reset_index(drop=True)
BLDG = pd.Index(meta[mask]['bldg_id'].values, name='bldg_id')

X_sub.set_index(BLDG).to_parquet(DATA_PROCESSED / 'X_47features.parquet')
COLS_BASE = list(X_sub.columns)

print(f'{len(X_sub):,} logements | {X_sub.shape[1]} features de base | '
      f'{ashrae_sub.nunique()} zones ASHRAE')
for k in TARGETS:
    print(f'  {k:12} : {(Yt_sub[k] > 0).mean()*100:5.1f} % de logements > 0'
          f'   (moyenne {Yt_sub[k].mean():.0f} kWh/an)')

In [ ]:
eff  = meta[mask]['in.hvac_heating_efficiency'].astype(str).reset_index(drop=True)
hspf = eff.str.extract(r'([\d.]+)\s*HSPF')[0].astype(float)

X_sub['pac']           = hspf.notna().astype(int)
X_sub['cop_chauffage'] = np.where(hspf.notna(), hspf / 3.412, 1.0)
X_sub['mshp']          = eff.str.startswith('MSHP').astype(int)

wea = pd.read_parquet(DATA_PROCESSED / 'weather_static.parquet').set_index('in.county')
wc  = wea.reindex(meta[mask]['in.county'].values).reset_index(drop=True)
wc  = wc.fillna(wc.median(numeric_only=True))
for c in wc.columns:
    X_sub['w_' + c] = wc[c].values

X_sub['UA_x_HDD'] = X_sub['UA'] * X_sub['w_HDD18']
X_sub['UA_x_CDD'] = X_sub['UA'] * X_sub['w_CDD18']

X_sub = to_numpy_dtypes(X_sub)
X_sub.set_index(BLDG).to_parquet(DATA_PROCESSED / 'X_features_v2.parquet')

BLOC_CHAUFFAGE = ['pac', 'cop_chauffage', 'mshp']
BLOC_CLIMAT    = [c for c in X_sub.columns if c.startswith('w_')] + ['UA_x_HDD', 'UA_x_CDD']
BLOC_ENVELOPPE = ['UA', 'H_ve', 'C', 'A_solaire', 'compacite', 'DSE']

print(f'{X_sub.shape[1]} features : {len(COLS_BASE)} de base '
      f'+ {len(BLOC_CHAUFFAGE)} chauffage + {len(BLOC_CLIMAT)} climat')
print(f"pompes à chaleur : {X_sub['pac'].mean()*100:.0f} % du parc | "
      f"COP médian {X_sub.loc[X_sub['pac'] == 1, 'cop_chauffage'].median():.2f}")

In [ ]:
z = ashrae_sub.copy()
z = z.where(~z.isin(z.value_counts()[lambda s: s < 5].index), 'RARE')

i_tv, i_te = train_test_split(np.arange(len(X_sub)), test_size=.2,
                              random_state=SEED, stratify=z)
i_tr, i_va = train_test_split(i_tv, test_size=.2, random_state=SEED, stratify=z.iloc[i_tv])

X_train, X_val, X_test = X_sub.iloc[i_tr], X_sub.iloc[i_va], X_sub.iloc[i_te]
Yt_train, Yt_val, Yt_test = Yt_sub.iloc[i_tr], Yt_sub.iloc[i_va], Yt_sub.iloc[i_te]
zone_test = ashrae_sub.iloc[i_te].values

print(f'train {len(i_tr):,} | val {len(i_va):,} | test {len(i_te):,}')

In [ ]:
best_par = {}
for usage in TARGETS:
    def objectif(t):
        p = params(usage, dict(
            learning_rate=t.suggest_float('learning_rate', .005, .08, log=True),
            num_leaves=t.suggest_int('num_leaves', 20, 200),
            min_child_samples=t.suggest_int('min_child_samples', 10, 400),
            colsample_bytree=t.suggest_float('colsample_bytree', .4, 1.),
            subsample=t.suggest_float('subsample', .5, 1.),
            reg_lambda=t.suggest_float('reg_lambda', .1, 100., log=True),
            reg_alpha=t.suggest_float('reg_alpha', .01, 20., log=True)), arbres=3000)
        m = lgb.LGBMRegressor(**p)
        m.fit(X_train, Yt_train[usage], eval_set=[(X_val, Yt_val[usage])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        return np.sqrt(mean_squared_error(Yt_val[usage], m.predict(X_val)))

    st = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
    st.optimize(objectif, n_trials=N_TRIALS, show_progress_bar=False)
    best_par[usage] = st.best_params
    print(f'{usage:11} val RMSE {st.best_value:8.1f} kWh')

pd.DataFrame(best_par).round(4).to_string()

In [ ]:
modeles, preds = {}, {}
lignes = []
for usage in TARGETS:
    m, p = entrainer(usage, X_sub.columns, best_par[usage])
    modeles[usage], preds[usage] = m, p
    y = Yt_test[usage]
    lignes.append({'usage': usage, 'R2': r2_score(y, p),
                   'RMSE': np.sqrt(mean_squared_error(y, p)),
                   'MAE': mean_absolute_error(y, p),
                   'RMSE_%': np.sqrt(mean_squared_error(y, p)) / y.mean() * 100,
                   'objectif': 'tweedie' if usage in TWEEDIE else 'l2',
                   'arbres': m.best_iteration_})
res = pd.DataFrame(lignes).set_index('usage').round(3)
print(res.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, usage in zip(axes.ravel(), TARGETS):
    y, p = Yt_test[usage].values, preds[usage]
    hi = np.percentile(np.r_[y, p], 99.5)
    ax.scatter(y, p, s=5, alpha=.15, color=REEL, edgecolors='none')
    ax.plot([0, hi], [0, hi], 'k--', lw=1)
    ax.set(xlim=(0, hi), ylim=(0, hi), xlabel='réel (kWh/an)', ylabel='prédit (kWh/an)',
           title=f"{usage} — R² {res.loc[usage, 'R2']:.3f}   "
                 f"RMSE {res.loc[usage, 'RMSE']:.0f} kWh")
    ax.grid(alpha=.25)
plt.suptitle(f'Réel vs prédit par usage — {len(i_te):,} logements de test', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
imp = pd.Series(modeles['chauffage'].feature_importances_,
                index=X_sub.columns).sort_values(ascending=False).head(20).sort_values()


def couleur(f):
    if f in BLOC_CHAUFFAGE:
        return PREDIT
    if f in BLOC_CLIMAT:
        return '#5F7C2E'
    if f in BLOC_ENVELOPPE:
        return REEL
    return NEUTRE


fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp.index, imp.values, color=[couleur(f) for f in imp.index])
ax.set(xlabel='gain LightGBM', title='Importance des variables — usage chauffage (top 20)')
ax.grid(axis='x', alpha=.3)
poignees = [plt.Rectangle((0, 0), 1, 1, color=c) for c in [PREDIT, '#5F7C2E', REEL, NEUTRE]]
ax.legend(poignees, ['système de chauffage', 'climat', 'enveloppe agrégée', 'autres'],
          loc='lower right')
plt.tight_layout(); plt.show()

for nom, bloc in [('chauffage', BLOC_CHAUFFAGE), ('climat', BLOC_CLIMAT),
                  ('enveloppe', BLOC_ENVELOPPE)]:
    tot = pd.Series(modeles['chauffage'].feature_importances_, index=X_sub.columns)
    print(f'{nom:12} {100 * tot[bloc].sum() / tot.sum():5.1f} % du gain total')

In [ ]:
err = pd.DataFrame({'zone': zone_test, 'reel': Yt_test['total'].values, 'pred': preds['total']})
err['biais'] = err['pred'] - err['reel']
err['abs'] = err['biais'].abs()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
g = err.groupby('zone').agg(rmse=('biais', lambda x: np.sqrt((x**2).mean())),
                            n=('reel', 'count'))
g = g[g['n'] >= 20].sort_values('rmse')
ax.barh(range(len(g)), g['rmse'], color=REEL)
ax.set_yticks(range(len(g)))
ax.set_yticklabels([f'{i} (n={n})' for i, n in zip(g.index, g['n'])], fontsize=9)
ax.axvline(np.sqrt((err['biais']**2).mean()), color='black', ls='--', lw=1)
ax.set(xlabel='RMSE (kWh/an)', title='Erreur par zone climatique')
ax.grid(axis='x', alpha=.3)

ax = axes[1]
ax.scatter(err['reel'], err['biais'], s=5, alpha=.12, color=REEL, edgecolors='none')
ax.axhline(0, color='black', lw=1)
ax.set(xlabel='consommation réelle (kWh/an)', ylabel='biais prédit − réel (kWh/an)',
       xlim=(0, np.percentile(err['reel'], 99.5)),
       title='Biais en fonction de la taille du logement')
ax.grid(alpha=.25)

ax = axes[2]
q = pd.qcut(err['reel'], 10, labels=False, duplicates='drop')
med = err.groupby(q)['biais'].median()
p25 = err.groupby(q)['biais'].quantile(.25)
p75 = err.groupby(q)['biais'].quantile(.75)
ax.fill_between(med.index + 1, p25, p75, color=REEL, alpha=.2)
ax.plot(med.index + 1, med, 'o-', color=REEL, lw=2)
ax.axhline(0, color='black', lw=1)
ax.set(xlabel='décile de consommation', ylabel='biais médian (kWh/an)',
       title='Biais médian par décile')
ax.grid(alpha=.25)

plt.suptitle('Diagnostic de l\'erreur — cible total', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
if COMPARER:
    avant = {}
    for usage in TARGETS:
        _, p = entrainer(usage, COLS_BASE, best_par[usage])
        avant[usage] = r2_score(Yt_test[usage], p)

    comp = pd.DataFrame({'avant': avant, 'après': res['R2']})
    comp['gain'] = comp['après'] - comp['avant']
    print(comp.round(4).to_string())

    fig, ax = plt.subplots(figsize=(8, 4))
    y = np.arange(len(comp))
    ax.barh(y + .19, comp['avant'], height=.34, color=NEUTRE, label=f'{len(COLS_BASE)} features')
    ax.barh(y - .19, comp['après'], height=.34, color=REEL, label=f'{X_sub.shape[1]} features')
    for k, (a, b) in enumerate(zip(comp['avant'], comp['après'])):
        ax.text(a + .005, k + .19, f'{a:.3f}', va='center', fontsize=9, color='#40525A')
        ax.text(b + .005, k - .19, f'{b:.3f}', va='center', fontsize=9)
    ax.set_yticks(y); ax.set_yticklabels(comp.index); ax.invert_yaxis()
    ax.set(xlim=(0, 1.1), xlabel='R² (test)',
           title='Apport du système de chauffage et du climat')
    ax.legend(loc='lower right'); ax.grid(axis='x', alpha=.3)
    plt.tight_layout(); plt.show()

In [ ]:
K   = 5
kf  = KFold(n_splits=K, shuffle=True, random_state=SEED)
oos = pd.DataFrame(index=BLDG, columns=list(TARGETS), dtype='float64')

for k, (i_ap, i_pr) in enumerate(kf.split(X_sub), 1):
    i_ap2, i_va2 = train_test_split(i_ap, test_size=.15, random_state=SEED)
    for usage in TARGETS:
        m = lgb.LGBMRegressor(**params(usage, best_par[usage]))
        m.fit(X_sub.iloc[i_ap2], Yt_sub[usage].iloc[i_ap2],
              eval_set=[(X_sub.iloc[i_va2], Yt_sub[usage].iloc[i_va2])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oos.iloc[i_pr, list(TARGETS).index(usage)] = np.clip(m.predict(X_sub.iloc[i_pr]), 0, None)
    print(f'  fold {k}/{K}')

oos.to_parquet(DATA_PROCESSED / 'static_preds_oos.parquet')

print()
for usage in TARGETS:
    print(f'{usage:11} R² hors échantillon {r2_score(Yt_sub[usage], oos[usage]):.4f}'
          f'   (test {res.loc[usage, "R2"]:.4f})')
print('\nExport -> static_preds_oos.parquet')